# Lab 09 · Evaluation: a multi-agent system against a single model

**Prepared by Mustafa Assaf** · Self-study: Go deeper · 15–20 minute lab

## Evaluate a multi-agent system against a baseline

**So far:** Lab 08 scored the council's option on held-out cases, and Lab 06's council made five model calls for one decision.

**New:** an *experiment* that asks whether a multi-agent system is worth its extra calls: it compares the council with each of its models working alone, the *baseline*. Every run is logged to **MLflow**: parameters, metrics, and model calls appear in the MLflow UI, where you can compare systems side by side. In the course project, this is the first experiment. Four measures compare them:

1. **Correct:** the option matches the expert's decision.
2. **Grounded:** the answer cites sources, and every cited source exists.
3. **Names a cost:** the answer states a downside of its own option. This needs judgement, so an *LLM judge* rates it.
4. **Cost:** model calls and seconds.

MLflow and LangSmith serve different roles. LangSmith traces *how the system runs* — the tree of agent calls, their inputs, outputs, and timings. MLflow tracks *what was learned* — which configuration beat which, with which metrics. One is a debugger for the run; the other is a logbook for the experiment.

An LLM judge rates many answers quickly, but a judge can be wrong. So we trust it only after it agrees with a human architect on a sample of answers. This is *calibration*.

The answers were recorded in earlier runs, as you would load them from your project's logs; `coursekit`'s `summarize_trace` reports the calls and seconds of a live run. Only the judge calls a model in this lab.

**At a glance**

![Lab 09 at a glance: Recorded runs (4 systems × 2 cases), then Correct · grounded · calls · seconds (logged to MLflow), then LLM judge (names a cost?), then Calibrate (agree with human ≥ 75%), then Council vs best (wins, and at what cost).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/09.svg)

## The idea in context

A council is more complex than one model and makes several times as many calls, so it must earn its place. The honest test is a controlled comparison: run the council and each single model on the same scenarios, and measure several things, including cost and time, not only correctness.

Some qualities, such as "does the answer name a cost?", need judgement. An LLM judge (Zheng and colleagues, 2023) uses a model to rate answers at scale, but judges have biases, for example towards longer answers or towards their own model family. So the judge is calibrated first: compared with a human's ratings on a sample, and trusted only if it agrees often enough.

| Key term | Meaning |
| --- | --- |
| **Baseline** | The system to beat; here, the best single model. |
| **Metric** | A number that measures one quality, such as correct answers or model calls. |
| **LLM judge** | A model that rates other models' answers against a rubric. |
| **Calibration** | Checking the judge against human ratings before trusting it. |

## Pattern · Evaluating a pattern

This lab is not a pattern of its own. It measures the quality attributes that the pattern cards in Labs 06–08 name, such as Accuracy, Cost, and Latency, for *Voting-Based Cooperation* and *Debate-Based Cooperation*. The LLM judge is one of the aggregation options that the field guide lists for Voting-Based Cooperation; here it rates answers, and it is trusted only after calibration.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
if not find_spec('mlflow'):
    %pip install -q mlflow
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)


## Step 1 · Recorded answers and the expert's decisions

The scenarios are Lab 08's two held-out cases, with the expert's decision for each. Each run records the option, the cited sources, the answer, the model calls, and the seconds it took. The source ids are R1 for the policy, C1 for the case's team facts, C2 for its operations facts, and M1 for a lesson from memory.

In [ ]:
import json

SOURCES = {"R1", "C1", "C2", "M1"}
expert = {"Logistics": "modular monolith", "Search": "microservices"}
runs = {  # (system, scenario): (option, cited sources, answer, model calls, seconds)
    ("Qwen", "Logistics"): ("microservices", ["R1"], "Two squads that release alone meet R1.", 1, 1.6),
    ("Qwen", "Search"): ("microservices", ["R1", "C1"], "Three squads that release alone meet R1.", 1, 1.5),
    ("DeepSeek", "Logistics"): ("modular monolith", ["C2"], "Squads lose separate releases; one operator copes.", 1, 2.2),
    ("DeepSeek", "Search"): ("microservices", [], "Microservices scale better.", 1, 2.0),
    ("Gemini", "Logistics"): ("microservices", [], "Microservices are the modern choice.", 1, 1.9),
    ("Gemini", "Search"): ("microservices", ["R1", "C2"], "R1 holds; on-call work grows for four operators.", 1, 2.1),
    ("Council", "Logistics"): ("modular monolith", ["C2", "M1"], "One operator; squads lose separate releases.", 5, 9.8),
    ("Council", "Search"): ("microservices", ["R1", "C2"], "R1 holds, but on-call work grows.", 5, 9.1),
}
systems = ["Qwen", "DeepSeek", "Gemini", "Council"]

## Step 2 · Measures that need no judge

`measure` counts one system's correct and grounded answers, and adds up its calls and seconds. The assert checks that every system answered every scenario, because a comparison is only fair on the same scenarios.

In [ ]:
import mlflow

mlflow.set_tracking_uri("file:///tmp/mlruns")

def measure(system):
    """Correct and grounded answers, calls, and seconds for one system."""
    rows = [(scenario, run) for (s, scenario), run in runs.items() if s == system]
    return {"correct": sum(run[0] == expert[scenario] for scenario, run in rows),
            "grounded": sum(bool(run[1]) and set(run[1]) <= SOURCES for _, run in rows),
            "calls": sum(run[3] for _, run in rows), "seconds": round(sum(run[4] for _, run in rows), 1)}

with mlflow.start_run(run_name="evaluate"):
    scores = {system: measure(system) for system in systems}
    for system, s in scores.items():
        with mlflow.start_run(run_name=system, nested=True):
            mlflow.log_params({"system": system, "scenarios": list(expert.keys())})
            mlflow.log_metrics(s)
        print(f"{system:<8} correct {s['correct']}/2  grounded {s['grounded']}/2  "
              f"calls {s['calls']:>2}  seconds {s['seconds']}")


## Step 3 · An LLM judge

The judge gets a rubric with one yes-or-no question, the chosen option, and the answer. A narrow question is easier to judge, and to check, than a score from 1 to 10. In practice mode, `judge_script` holds the judge's verdicts, in the order of `runs`.

A judge from the same model family as one of the systems may favour that system. In your project, prefer a judge from another family, or check its agreement with the human for each system.

In [ ]:
RUBRIC = ("Does this architecture answer name at least one cost or downside of the option it chose? "
          'Reply with JSON only, no code fences: {"names_cost": true or false}')
judge_script = [False, False, True, True, False, True, True, True]

def judge(option, answer, example):
    """One yes-or-no verdict from the judge model."""
    reply = ask(f"{RUBRIC}\nChosen option: {option}\nAnswer: {answer}", example=json.dumps({"names_cost": example}))
    return json.loads(reply)["names_cost"] is True

judged = {key: judge(run[0], run[2], example) for (key, run), example in zip(runs.items(), judge_script)}
for (system, scenario), verdict in judged.items():
    print(f"{system:<8} {scenario:<10} names a cost: {verdict}")

## Step 4 · Calibrate the judge

A human architect rated four of the answers. The judge's verdicts count only if they agree with the human on at least 75% of this sample. Every disagreement is printed, so you can read why the judge erred.

In [ ]:
human = {("Qwen", "Logistics"): False, ("DeepSeek", "Logistics"): True,  # a human architect's ratings of a sample
         ("DeepSeek", "Search"): False, ("Gemini", "Search"): True}
agree = sum(judged[key] == rating for key, rating in human.items())
trusted = agree / len(human) >= 0.75
print(f"The judge agrees with the human on {agree} of {len(human)} sampled answers. Trusted: {trusted}")
for key, rating in human.items():
    if judged[key] != rating:
        print(f"  {key}: {runs[key][2]!r} -> judge says {judged[key]}, human says {rating}")

## Step 5 · The council against the best single model

The baseline is the strongest single model, chosen by correct answers and then by grounded ones. An untrusted judge's measure is reported as `None` and never compared. The last line names the measures on which the council wins, and its cost.

In [ ]:
with mlflow.start_run(run_name="council-vs-best"):
    for system in systems:
        scores[system]["names_cost"] = sum(v for (s, _), v in judged.items() if s == system) if trusted else None
    best = max(systems[:3], key=lambda s: (scores[s]["correct"], scores[s]["grounded"]))  # the strongest baseline
    for metric in ("correct", "grounded", "names_cost", "calls", "seconds"):
        print(f"{metric:<11} council {scores['Council'][metric]!s:<6} {best} alone {scores[best][metric]}")
    wins = [m for m in ("correct", "grounded", "names_cost")
            if scores["Council"][m] is not None and scores["Council"][m] > scores[best][m]]
    ratio = scores["Council"]["calls"] / scores[best]["calls"]
    mlflow.log_params({"baseline": best, "scenarios": list(expert.keys())})
    mlflow.log_metrics({
        "council_correct": scores["Council"]["correct"],
        "council_grounded": scores["Council"]["grounded"],
        "council_names_cost": scores["Council"]["names_cost"] or 0,
        "council_calls": scores["Council"]["calls"],
        "council_seconds": scores["Council"]["seconds"],
        f"{best}_correct": scores[best]["correct"],
        f"{best}_grounded": scores[best]["grounded"],
        f"{best}_calls": scores[best]["calls"],
        f"{best}_seconds": scores[best]["seconds"],
        "council_wins": len(wins),
        "calls_ratio": round(ratio, 2),
    })
    print(f"\nThe council beats {best} on {wins or 'nothing'}, with {ratio:.0f} times the model calls.")


## What you should see

Open the MLflow UI with:

```bash
mlflow ui --port 5000
```

Then open `localhost:5000` and navigate to the **evaluate** experiment. You will see four nested runs, one per system, each showing its logged parameters and metrics.

In Step 2, Qwen and Gemini each get 1 of 2 correct, and DeepSeek and the council get both. DeepSeek and Gemini each have one answer that cites no source. The council made 10 calls, and each single model 2. Step 3 prints eight verdicts. In Step 4, the judge agrees with the human on 3 of 4 answers: it says that DeepSeek's "Microservices scale better." names a cost, but the human says it names only a benefit. 3 of 4 meets the 75% threshold, so the judge is trusted.

Step 5 picks DeepSeek as the baseline. The council ties DeepSeek on correct answers and on named costs, and beats it on grounding, with 5 times the calls. Look at the tie on named costs: it comes from the judge's one mistake. By the human's ratings, the council names a cost in both answers, and DeepSeek in one.

The last cell in the MLflow UI, under the `council-vs-best` experiment run, shows the side-by-side comparison logged as metrics.
